In [100]:
! pip install pandas -q
! pip install numpy -q
! pip install scikit-learn -q

You should consider upgrading via the 'C:\Users\bt984\Desktop\AI Wealth Advisor\backend\env\Scripts\python.exe -m pip install --upgrade pip' command.
You should consider upgrading via the 'C:\Users\bt984\Desktop\AI Wealth Advisor\backend\env\Scripts\python.exe -m pip install --upgrade pip' command.
You should consider upgrading via the 'C:\Users\bt984\Desktop\AI Wealth Advisor\backend\env\Scripts\python.exe -m pip install --upgrade pip' command.


In [101]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder, StandardScaler


In [102]:
data = pd.read_csv('risk_training_data.csv')
data

,age,income_usd,savings_usd,time_horizon,risk_tolerance,goal,stocks,etfs,bonds,crypto,cash,risk_score,risk_label,expected_return_pct
0,36,38000,8000,5,high,house,56.1,21.9,7.0,10.0,5.0,94.3,High,13.2
1,23,27000,4000,20,medium,house,49.6,24.7,13.8,4.9,7.0,58.2,Medium,7.5
2,43,52000,21000,10,low,retirement,24.2,20.1,35.8,1.8,18.1,24.1,Low,5.5
3,42,55000,14500,5,high,wealth_building,54.5,22.1,8.5,9.9,5.0,89.7,High,13.9
4,33,46000,16000,15,medium,retirement,46.9,24.9,16.3,5.0,6.9,51.1,Medium,9.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,61,74000,72500,2,medium,retirement,29.4,25.6,31.8,0.8,12.4,51.1,Medium,7.1
996,52,62000,61000,3,high,travel,42.0,22.3,19.2,6.4,10.1,73.1,High,12.3
997,29,33000,8000,20,high,retirement,57.9,21.9,5.3,9.9,5.0,82.0,High,13.4
998,22,36000,6500,20,low,wealth_building,30.1,19.8,30.4,2.0,17.7,27.0,Low,6.1


In [103]:
# Check dataset details
def check_dataset_detail (data):
    print("Dataset shape:", data.shape)
    print("Dataset columns:", data.columns)
    print("Dataset info:", data.info())
    print("Dataset description:", data.describe())
    
check_dataset_detail(data)

Dataset shape: (1000, 14)
Dataset columns: Index(['age', 'income_usd', 'savings_usd', 'time_horizon', 'risk_tolerance',
       'goal', 'stocks', 'etfs', 'bonds', 'crypto', 'cash', 'risk_score',
       'risk_label', 'expected_return_pct'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  1000 non-null   int64  
 1   income_usd           1000 non-null   int64  
 2   savings_usd          1000 non-null   int64  
 3   time_horizon         1000 non-null   int64  
 4   risk_tolerance       1000 non-null   object 
 5   goal                 1000 non-null   object 
 6   stocks               1000 non-null   float64
 7   etfs                 1000 non-null   float64
 8   bonds                1000 non-null   float64
 9   crypto               1000 non-null   float64
 10  cash                 1000 no

In [104]:
label_encoder = LabelEncoder()
scaler = StandardScaler()

In [105]:
data["risk_tolerance"].value_counts()

risk_tolerance
medium    548
high      245
low       207
Name: count, dtype: int64

In [106]:
data["goal"].value_counts()

goal
retirement             422
house                  197
wealth_building        156
children_education      78
passive_income          49
education               39
wealth_preservation     31
travel                  28
Name: count, dtype: int64

In [107]:
# preprocessing feature columns

def preprocessing_dataset(data, label_encoder, scaler):

  # drop columns
  data = data.drop(['risk_label', 'expected_return_pct', 'risk_score', 'income_usd', 'savings_usd', 'goal'], axis=1)

  num_cols = data.select_dtypes(exclude=['object']).columns 
  obj_cols = data.select_dtypes(include=['object']).columns 

  for col in obj_cols:
        data[col] = label_encoder.fit_transform(data[col])  #encoded columns

  data[num_cols]  = scaler.fit_transform(data[num_cols])

  return data

preprocessed_dataset = preprocessing_dataset(data.copy(), label_encoder, scaler)
print(preprocessed_dataset.iloc[:5, :])

        age  time_horizon  risk_tolerance    stocks      etfs     bonds  \
0 -0.397774     -1.029191               0  1.194943 -0.679146 -1.263640   
1 -1.547155      0.588525               2  0.629275  0.669608 -0.603091   
2  0.221123     -0.489952               1 -1.581181 -1.546202  1.533980   
3  0.132709     -1.029191               0  1.055702 -0.582806 -1.117931   
4 -0.663016      0.049286               2  0.394305  0.765948 -0.360242   

     crypto      cash  
0  1.672181 -0.849043  
1 -0.054369 -0.449672  
2 -1.103841  1.766838  
3  1.638327 -0.849043  
4 -0.020515 -0.469640  


In [108]:
y = data['risk_score'].values
# y = scaler.fit_transform(y)
# y = y.ravel() #Flatten to 1D
y[:10]

array([94.3, 58.2, 24.1, 89.7, 51.1, 58.2, 53. , 61. , 93.4, 50.9])

In [109]:
X = preprocessed_dataset
y

array([ 94.3,  58.2,  24.1,  89.7,  51.1,  58.2,  53. ,  61. ,  93.4,
        50.9,  13. ,  27.7,  29.4,  51.3,  50.1,  61.9,  58. ,  51.9,
        25.2,  98.7,  93.5,  60.2,  56. ,  82.6,  30.2,  57.4,  65.1,
        44.5,  58.1,  80.6,  21.8,  17.2,  60.2,  61.6,  29.7,  89.8,
        65.4,  69.4,  67.7,  68. ,  33.9,  16.7,  82.9,  60.6,  95.1,
        85.3,  91.2,  16.2,  14.5,  67.2,  52.4,  16.7,  69.2,  93.1,
        23.6,  98.8,  37.8,  53.3,  82.2,  78.6,  82.6,  18.6,  25.8,
        62.7,  24.9,   0. ,  92.3,  61.1,  53.1,  57.2,  62.6,  57.7,
        52.8,  62.7,  87.5,  24.8,  78.3,  57.2,  62.7,  22.3,  55.8,
        54.7,  28.8,  29.2,  91.6,  67.1,  60.1,  27.5,   2.6,  93.5,
        22.6,  51.8, 100. ,  28. ,  79.9,  63.6,  44.2,  53.4,  64. ,
        92.9,  85.7,  83.8,  16.5,  53.5,  51.6,  67.6,  56.2,  59.9,
        49.7,  92.6,  95.6,  63.2,  73.6,  20.1,  58.4,  51.4,  10. ,
        93.4,  51.4,  64.2,  59.1,  54.6,  58.6,  53.9,  50.8,  58.3,
        50.3,  61.9,

In [110]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [111]:
model = DecisionTreeRegressor(
        max_depth=8,          # limit depth to prevent overfitting
        min_samples_leaf=10,  # each leaf needs at least 10 samples
        random_state=42,
    )
    
model.fit(X_train, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,8
,min_samples_split,2
,min_samples_leaf,10
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [112]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae:.2f} points (out of 100)")
print(f"That means predictions are off by ~{mae:.0f} points on average")

Mean Absolute Error: 3.72 points (out of 100)
That means predictions are off by ~4 points on average


In [113]:
from sklearn.metrics import mean_squared_error, r2_score

# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Evaluate
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("Train MSE:", train_mse)
print("Test MSE:", test_mse)
print("Train R²:", train_r2)
print("Test R²:", test_r2)

Train MSE: 17.205882253430342
Test MSE: 22.76797439575769
Train R²: 0.9659545259886095
Test R²: 0.9580181944796095
